In [56]:
import pandas as pd

In [57]:
tourism_data = pd.read_csv('processed_data/processed_tourism.csv')
# destinationISO, originISO, Destination, Origin, Region, Subregion \
# Measure, Basis, Source table, Note \
# 2000 - 2020 \ measure_varies

trade_data = pd.read_csv('processed_data/comtrade_gdp.csv')
# refYear, reporterISO, partnerISO, fobvalue, reporterGDP, partnerGDP


In [58]:
# Set tourism data to be merged with trade data
year_cols = [c for c in tourism_data.columns if str(c).isdigit()]
long_tourism_data = tourism_data.melt(id_vars=["destinationISO", "originISO", "Measure"], value_vars=year_cols, var_name="year", value_name="arrivals")
long_tourism_data["year"] = long_tourism_data["year"].astype(int)

# Drop rows where destination = arrivals
long_tourism_data = long_tourism_data.dropna(subset=["destinationISO", "originISO", "arrivals"])
long_tourism_data = long_tourism_data[long_tourism_data["destinationISO"] != long_tourism_data["originISO"]]

# If printed is 0, there are no duplicate overlap years and pairs (good thing)
print(long_tourism_data.duplicated(["year", "destinationISO", "originISO"]).sum())

0


In [59]:
# Strip whitespace from ISO codes
for c in ["reporterISO", "partnerISO"]:
    trade_data[c] = trade_data[c].str.strip()
for c in ["destinationISO", "originISO"]:
    long_tourism_data[c] = long_tourism_data[c].str.strip()

# Recode trade's S19 (Taiwan) to TWN
trade_data["reporterISO"] = trade_data["reporterISO"].replace("S19", "TWN")
trade_data["partnerISO"] = trade_data["partnerISO"].replace("S19", "TWN")

# Recode trade's SCG to SRB (Serbia)
trade_data["reporterISO"] = trade_data["reporterISO"].replace("SCG", "SRB")
trade_data["partnerISO"] = trade_data["partnerISO"].replace("SCG", "SRB")

# Drop unnecessary ISO codes
tourism_drop_codes = ["GLP", "GUF", "MTQ", "REU", "PRI", "VIR", "IMN", "LIE", "SJM"]
trade_drop_codes = ["W00", "A49", "A59", "A79", "E19", "E29", "F19", "F49", "F97", "O19", "R20", "R91", "XX", "_X", "X1", "X2", "NTZ", "ZA1", "ATA", "ATB", "ATF", "BVT", "HMD", "SGS", "PUS", "BLM", "MYT", "ANT"]
long_tourism_data = long_tourism_data[~long_tourism_data["destinationISO"].isin(tourism_drop_codes)]
long_tourism_data = long_tourism_data[~long_tourism_data["originISO"].isin(tourism_drop_codes)]
trade_data = trade_data[~trade_data["reporterISO"].isin(trade_drop_codes)]
trade_data = trade_data[~trade_data["partnerISO"].isin(trade_drop_codes)]

# Check that tourism and trade codes share the same set
trade_codes = set(trade_data["reporterISO"]) | set(trade_data["partnerISO"])
tourism_codes = set(long_tourism_data["destinationISO"]) | set(long_tourism_data["originISO"])
print("In tourism, not trade:", sorted(tourism_codes - trade_codes))
print("In trade, not tourism:", sorted(trade_codes - tourism_codes))

In tourism, not trade: []
In trade, not tourism: []


In [64]:
# Merge datasets
keys = ["refYear", "destination_ISO", "origin_ISO"]
inbound_tourists = long_tourism_data.rename(columns={
    "year": "refYear", "destinationISO": "destination_ISO", "originISO": "origin_ISO", "arrivals": "origin_tourists", "Measure": "measure_in"
})
outbound_tourists = long_tourism_data.rename(columns={
    "year": "refYear", "destinationISO": "origin_ISO", "originISO": "destination_ISO", "arrivals": "destination_tourists", "Measure": "measure_out"
})
final_trade_data = trade_data.rename(columns={
    "reporterISO": "destination_ISO", "partnerISO": "origin_ISO"
})

final_trade_data = (final_trade_data
    .groupby(keys, as_index=False)
    .agg(fobvalue=("fobvalue", lambda s: s.sum(min_count=1)),
         reporterGDP=("reporterGDP", "first"),
         partnerGDP=("partnerGDP", "first")))

merged_data = (final_trade_data.merge(inbound_tourists[keys + ["origin_tourists", "measure_in"]], on=keys, how="left", validate="1:1").merge(outbound_tourists[keys + ["destination_tourists", "measure_out"]], on=keys, how="left", validate="1:1"))

# print(merged_data.groupby("refYear")[["origin_tourists", "destination_tourists"]].apply(lambda x: x.notna().mean()))

In [67]:
# Data verification
# This cell was made using Claude

tour_cols = ["origin_tourists", "destination_tourists"]
yr_min, yr_max = long_tourism_data["year"].min(), long_tourism_data["year"].max()

def section(title):
    print(f"\n{'=' * 60}\n{title}\n{'=' * 60}")

# 1. Row count
section("1. Row count (should be equal)")
print(f"Trade before merge: {len(final_trade_data):,}")
print(f"After merge:        {len(merged_data):,}")
print("OK" if len(final_trade_data) == len(merged_data) else "MISMATCH: check for duplicate keys")

# 2. Tourism match rate
section("2. Share of rows with tourism data, by year")
print(f"Year dtypes: trade = {final_trade_data['refYear'].dtype}, "
      f"tourism = {long_tourism_data['year'].dtype}")
coverage = merged_data.groupby("refYear")[tour_cols].apply(lambda x: x.notna().mean()).round(3)
print(coverage.to_string())
low = coverage[(coverage.max(axis=1) < 0.01) & coverage.index.to_series().between(yr_min, yr_max)]
print("OK" if low.empty else f"WARNING: near-zero match in {list(low.index)}")

# 3. Year coverage
section("3. Year ranges")
print(f"Trade:   {merged_data['refYear'].min()}–{merged_data['refYear'].max()}")
print(f"Tourism: {yr_min}–{yr_max}")

# 4. Missing GDP
section("4. Missing GDP")
print((merged_data[["reporterGDP", "partnerGDP"]].isna().mean() * 100).round(2)
      .astype(str).add("% missing").to_string())
print("\nPartners most often missing GDP:")
print(merged_data.loc[merged_data["partnerGDP"].isna(), "origin_ISO"]
      .value_counts().head(10).to_string())

# 5. Zero and missing trade values
section("5. Trade values")
print(f"Zero fobvalue:    {(merged_data['fobvalue'] == 0).sum():,}")
print(f"Missing fobvalue: {merged_data['fobvalue'].isna().sum():,}")

# 6. Tourism measure switches
section("6. Destinations whose tourism measure changes over time")
switches = long_tourism_data.groupby("destinationISO")["Measure"].nunique()
switches = switches[switches > 1]
print(f"{len(switches)} destinations: {', '.join(switches.index)}" if len(switches) else "None")


1. Row count (should be equal)
Trade before merge: 546,138
After merge:        546,138
OK

2. Share of rows with tourism data, by year
Year dtypes: trade = int64, tourism = int64
         origin_tourists  destination_tourists
refYear                                       
1998               0.000                 0.000
1999               0.000                 0.000
2000               0.392                 0.398
2001               0.399                 0.401
2002               0.394                 0.391
2003               0.392                 0.386
2004               0.403                 0.398
2005               0.406                 0.407
2006               0.411                 0.402
2007               0.408                 0.401
2008               0.412                 0.403
2009               0.413                 0.408
2010               0.418                 0.414
2011               0.414                 0.405
2012               0.454                 0.450
2013               0.

In [68]:
merged_data = merged_data[merged_data["refYear"].between(yr_min, yr_max)]
merged_data.to_csv("processed_data/trade_tourism_merged.csv", index=False)